# CSV Analyzer Tool (Practical Project)
LLM-callable CSV analysis with data-quality checks.

| Action | What it does |
|---|---|
| `profile` | shape, per-column stats, missing %, outliers, quality warnings |
| `group_by` | aggregate a metric per category |
| `top_n` | top/bottom N rows by a numeric column |
| `filter` | rows matching a condition |
| `correlation` | strongest numeric correlations |

## 1. Imports, loader and helpers

In [13]:
import csv
import json
import os
import warnings

import numpy as np
import pandas as pd


class CSVAnalyzerError(Exception):
    """Validation / parsing problem with an LLM-friendly message."""


AGGS = ("sum", "mean", "count", "min", "max", "median")
OPS = ("eq", "ne", "gt", "gte", "lt", "lte", "contains")


def _clean(o):
    """Make pandas/numpy results JSON-safe (NaN -> None, numpy -> python)."""
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, pd.Timestamp):
        return o.isoformat()
    if o is None or isinstance(o, (str, int, bool)):
        return o
    if o is pd.NaT:
        return None
    return str(o)


def load_csv(path: str, max_rows: int = 200_000) -> pd.DataFrame:
    if not os.path.isfile(path):
        raise CSVAnalyzerError(f"File not found: {os.path.basename(path)}")
    if os.path.getsize(path) == 0:
        raise CSVAnalyzerError("File is empty.")
    df = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, encoding=enc) as f:
                sample = f.read(4096)
            try:
                sep = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            except csv.Error:
                sep = ","
            df = pd.read_csv(path, sep=sep, encoding=enc, nrows=max_rows)
            break
        except UnicodeDecodeError:
            continue
        except pd.errors.EmptyDataError:
            raise CSVAnalyzerError("File has no data.")
        except pd.errors.ParserError as e:
            raise CSVAnalyzerError(f"Could not parse CSV: {e}")
    if df is None:
        raise CSVAnalyzerError("Could not decode file (tried utf-8 and latin-1).")
    df.columns = [str(c).strip() for c in df.columns]
    if df.empty:
        raise CSVAnalyzerError("CSV has headers but no rows.")
    return df


def _numeric_cols(df):
    return list(df.select_dtypes(include="number").columns)


def _need_col(df, col, label="column"):
    if col is None:
        raise CSVAnalyzerError(f"'{label}' is required for this action.")
    if col not in df.columns:
        raise CSVAnalyzerError(f"Column '{col}' not found. Available: {list(df.columns)}")
    return col


def _need_numeric(df, col):
    _need_col(df, col, "metric/column")
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise CSVAnalyzerError(f"Column '{col}' is not numeric (dtype {df[col].dtype}).")

## 2. Analysis actions

In [14]:
def profile(df: pd.DataFrame) -> dict:
    n = len(df)
    warns, cols = [], {}
    for c in df.columns:
        s = df[c]
        info = {"dtype": str(s.dtype), "missing": int(s.isna().sum()),
                "missing_pct": round(float(s.isna().mean()) * 100, 1), "unique": int(s.nunique())}
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            d = s.dropna()
            if len(d):
                q1, q3 = d.quantile(0.25), d.quantile(0.75)
                iqr = q3 - q1
                out = int(((d < q1 - 1.5 * iqr) | (d > q3 + 1.5 * iqr)).sum()) if iqr > 0 else 0
                info.update(min=d.min(), max=d.max(), mean=round(float(d.mean()), 2),
                            median=float(d.median()), std=round(float(d.std()), 2) if len(d) > 1 else 0.0,
                            outliers_iqr=out)
                if out:
                    warns.append(f"'{c}' has {out} outlier(s) (IQR rule).")
        else:
            info["top_values"] = s.value_counts().head(3).to_dict()
            nums = pd.to_numeric(s, errors="coerce")
            if s.notna().any() and nums.notna().sum() / max(s.notna().sum(), 1) >= 0.9:
                warns.append(f"'{c}' looks numeric but is stored as text - clean it before analysis.")
            else:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    dts = pd.to_datetime(s.dropna().head(50), errors="coerce")
                if len(dts) and dts.notna().mean() >= 0.9 and s.nunique() > 1:
                    info["looks_like_date"] = True
        if info["missing_pct"] >= 30:
            warns.append(f"'{c}' is {info['missing_pct']}% missing.")
        if info["unique"] <= 1 and n > 1:
            warns.append(f"'{c}' is constant (no information).")
        cols[c] = info
    dup = int(df.duplicated().sum())
    if dup:
        warns.append(f"{dup} duplicate row(s) found.")
    return {"rows": n, "columns_count": df.shape[1], "duplicate_rows": dup,
            "columns": cols, "warnings": warns}


def group_by(df, by, metric, agg="sum", top=20) -> dict:
    _need_col(df, by, "by")
    if agg not in AGGS:
        raise CSVAnalyzerError(f"agg must be one of {AGGS}.")
    if agg != "count":
        _need_numeric(df, metric)
    else:
        _need_col(df, metric, "metric")
    res = df.groupby(by)[metric].agg(agg).sort_values(ascending=False).head(top)
    return {"by": by, "metric": metric, "agg": agg, "groups": res.round(2).to_dict()}


def top_n(df, column, n=5, ascending=False) -> dict:
    _need_numeric(df, column)
    n = max(1, min(int(n), 50))
    rows = df.sort_values(column, ascending=ascending).head(n)
    return {"column": column, "ascending": ascending, "rows": rows.to_dict("records")}


def filter_rows(df, column, op="eq", value=None, limit=20) -> dict:
    _need_col(df, column)
    if op not in OPS:
        raise CSVAnalyzerError(f"op must be one of {OPS}.")
    if value is None:
        raise CSVAnalyzerError("'value' is required for filter.")
    s = df[column]
    if op == "contains":
        mask = s.astype(str).str.contains(str(value), case=False, na=False)
    else:
        if pd.api.types.is_numeric_dtype(s):
            try:
                value = float(value)
            except (TypeError, ValueError):
                raise CSVAnalyzerError(f"Column '{column}' is numeric; value must be a number.")
        mask = {"eq": s == value, "ne": s != value, "gt": s > value,
                "gte": s >= value, "lt": s < value, "lte": s <= value}[op]
    hit = df[mask]
    limit = max(1, min(int(limit), 100))
    return {"matches": int(len(hit)), "returned": min(len(hit), limit), "rows": hit.head(limit).to_dict("records")}


def correlation(df, top=5) -> dict:
    nums = _numeric_cols(df)
    if len(nums) < 2:
        raise CSVAnalyzerError("Need at least 2 numeric columns for correlation.")
    corr = df[nums].corr()
    pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(nums) for b in nums[i + 1:] if pd.notna(corr.loc[a, b])]
    pairs.sort(key=lambda t: abs(t[2]), reverse=True)
    return {"top_pairs": [{"a": a, "b": b, "r": round(float(r), 3)} for a, b, r in pairs[:top]]}

## 3. Single entry point

In [15]:
def analyze(path: str, action: str = "profile", column=None, by=None, metric=None,
            agg="sum", n=5, op="eq", value=None, ascending=False) -> dict:
    df = load_csv(path)
    if action == "profile":
        out = profile(df)
    elif action == "group_by":
        out = group_by(df, by, metric, agg)
    elif action == "top_n":
        out = top_n(df, column, n, ascending)
    elif action == "filter":
        out = filter_rows(df, column, op, value)
    elif action == "correlation":
        out = correlation(df)
    else:
        raise CSVAnalyzerError("action must be one of: profile, group_by, top_n, filter, correlation.")
    return _clean(out)

## 4. Tool wrapper + function-calling schema
Every call returns the same envelope so an LLM (or a chain) can react instead of crashing:
- success: `{"ok": True, "data": {...}}`
- failure: `{"ok": False, "error": {"type", "message", "retryable"}}`

In [16]:
def csv_analyzer(**kwargs):
    """Tool entry point with a uniform success/error envelope."""
    try:
        if "path" not in kwargs:
            raise CSVAnalyzerError("Missing required argument: 'path'.")
        return {"ok": True, "data": analyze(**kwargs)}
    except CSVAnalyzerError as e:
        return {"ok": False, "error": {"type": "analysis_error", "message": str(e), "retryable": False}}
    except TypeError as e:
        return {"ok": False, "error": {"type": "invalid_arguments", "message": str(e), "retryable": False}}
    except Exception as e:
        return {"ok": False, "error": {"type": "internal_error", "message": f"{type(e).__name__}: {e}", "retryable": False}}


CSV_ANALYZER_SCHEMA = {
    "name": "csv_analyzer",
    "description": "Deep CSV analysis. action='profile' (default): stats, missing %, outliers, quality warnings. "
                   "Also: group_by (by+metric+agg), top_n (column,n), filter (column,op,value), correlation. "
                   "Call profile first on an unknown file.",
    "input_schema": {  # for OpenAI use {"type":"function","function":{... "parameters": <this schema>}}
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "CSV file path"},
            "action": {"type": "string", "enum": ["profile", "group_by", "top_n", "filter", "correlation"]},
            "column": {"type": "string"}, "by": {"type": "string"}, "metric": {"type": "string"},
            "agg": {"type": "string", "enum": list(AGGS)},
            "n": {"type": "integer"},
            "op": {"type": "string", "enum": list(OPS)},
            "value": {"type": "string"},
            "ascending": {"type": "boolean"},
        },
        "required": ["path"],
    },
}

## 5. Create sample data (messy on purpose)

In [17]:
import os, json
os.makedirs("data", exist_ok=True)
SAMPLE = "data/orders.csv"
with open(SAMPLE, "w", encoding="utf-8") as f:
    f.write("order_id,date,customer,category,amount,rating\n"
            "1,2026-10-01,Asha,Books,250,4\n2,2026-10-02,Ravi,Books,300,5\n3,2026-10-03,Asha,Games,1200,3\n"
            "4,2026-10-04,Meera,Games,1100,\n5,2026-10-05,Ravi,Books,280,4\n6,2026-10-06,Meera,Tech,45000,5\n"
            "7,2026-10-07,Asha,Tech,900,2\n7,2026-10-07,Asha,Tech,900,2\n")

def show(title, res):
    print(f"\n=== {title} ===")
    print(json.dumps(res, indent=2, default=str))

## 6. Demo: every action

In [18]:
res = csv_analyzer(path=SAMPLE)                      # profile (default)
show("profile - warnings", res["data"]["warnings"])
show("profile - columns", res["data"]["columns"])


=== profile - warnings ===
[
  "'amount' has 1 outlier(s) (IQR rule).",
  "1 duplicate row(s) found."
]

=== profile - columns ===
{
  "order_id": {
    "dtype": "int64",
    "missing": 0,
    "missing_pct": 0.0,
    "unique": 7,
    "min": 1,
    "max": 7,
    "mean": 4.38,
    "median": 4.5,
    "std": 2.26,
    "outliers_iqr": 0
  },
  "date": {
    "dtype": "str",
    "missing": 0,
    "missing_pct": 0.0,
    "unique": 7,
    "top_values": {
      "2026-10-07": 2,
      "2026-10-01": 1,
      "2026-10-02": 1
    },
    "looks_like_date": true
  },
  "customer": {
    "dtype": "str",
    "missing": 0,
    "missing_pct": 0.0,
    "unique": 3,
    "top_values": {
      "Asha": 4,
      "Ravi": 2,
      "Meera": 2
    }
  },
  "category": {
    "dtype": "str",
    "missing": 0,
    "missing_pct": 0.0,
    "unique": 3,
    "top_values": {
      "Books": 3,
      "Tech": 3,
      "Games": 2
    }
  },
  "amount": {
    "dtype": "int64",
    "missing": 0,
    "missing_pct": 0.0,
    "uni

In [19]:
show("group_by category", csv_analyzer(path=SAMPLE, action="group_by", by="category", metric="amount", agg="sum"))


=== group_by category ===
{
  "ok": true,
  "data": {
    "by": "category",
    "metric": "amount",
    "agg": "sum",
    "groups": {
      "Tech": 46800,
      "Games": 2300,
      "Books": 830
    }
  }
}


In [20]:
show("top_n amount", csv_analyzer(path=SAMPLE, action="top_n", column="amount", n=2))


=== top_n amount ===
{
  "ok": true,
  "data": {
    "column": "amount",
    "ascending": false,
    "rows": [
      {
        "order_id": 6,
        "date": "2026-10-06",
        "customer": "Meera",
        "category": "Tech",
        "amount": 45000,
        "rating": 5.0
      },
      {
        "order_id": 3,
        "date": "2026-10-03",
        "customer": "Asha",
        "category": "Games",
        "amount": 1200,
        "rating": 3.0
      }
    ]
  }
}


In [21]:
show("filter amount > 1000", csv_analyzer(path=SAMPLE, action="filter", column="amount", op="gt", value="1000"))


=== filter amount > 1000 ===
{
  "ok": true,
  "data": {
    "matches": 3,
    "returned": 3,
    "rows": [
      {
        "order_id": 3,
        "date": "2026-10-03",
        "customer": "Asha",
        "category": "Games",
        "amount": 1200,
        "rating": 3.0
      },
      {
        "order_id": 4,
        "date": "2026-10-04",
        "customer": "Meera",
        "category": "Games",
        "amount": 1100,
        "rating": null
      },
      {
        "order_id": 6,
        "date": "2026-10-06",
        "customer": "Meera",
        "category": "Tech",
        "amount": 45000,
        "rating": 5.0
      }
    ]
  }
}


In [22]:
show("correlation", csv_analyzer(path=SAMPLE, action="correlation"))


=== correlation ===
{
  "ok": true,
  "data": {
    "top_pairs": [
      {
        "a": "order_id",
        "b": "rating",
        "r": -0.522
      },
      {
        "a": "amount",
        "b": "rating",
        "r": 0.48
      },
      {
        "a": "order_id",
        "b": "amount",
        "r": 0.3
      }
    ]
  }
}


## 7. Error handling demo

In [23]:
show("bad column",       csv_analyzer(path=SAMPLE, action="group_by", by="city", metric="amount"))
show("text metric",      csv_analyzer(path=SAMPLE, action="top_n", column="customer"))
show("bad operator",     csv_analyzer(path=SAMPLE, action="filter", column="amount", op="like", value="1"))
show("bad action",       csv_analyzer(path=SAMPLE, action="delete"))
show("missing file",     csv_analyzer(path="data/nope.csv"))
show("missing path arg", csv_analyzer(action="profile"))
open("data/empty.csv", "w").close()
show("empty file",       csv_analyzer(path="data/empty.csv"))


=== bad column ===
{
  "ok": false,
  "error": {
    "type": "analysis_error",
    "message": "Column 'city' not found. Available: ['order_id', 'date', 'customer', 'category', 'amount', 'rating']",
    "retryable": false
  }
}

=== text metric ===
{
  "ok": false,
  "error": {
    "type": "analysis_error",
    "message": "Column 'customer' is not numeric (dtype str).",
    "retryable": false
  }
}

=== bad operator ===
{
  "ok": false,
  "error": {
    "type": "analysis_error",
    "message": "op must be one of ('eq', 'ne', 'gt', 'gte', 'lt', 'lte', 'contains').",
    "retryable": false
  }
}

=== bad action ===
{
  "ok": false,
  "error": {
    "type": "analysis_error",
    "message": "action must be one of: profile, group_by, top_n, filter, correlation.",
    "retryable": false
  }
}

=== missing file ===
{
  "ok": false,
  "error": {
    "type": "analysis_error",
    "message": "File not found: nope.csv",
    "retryable": false
  }
}

=== missing path arg ===
{
  "ok": false,
  "er

## 8. Tool chaining
Chain: `profile` -> `group_by` -> compute top category share (what a `calculator` tool would do) -> draft a summary (what `email_tool` dry-run would do).
The chain stops at the first failed step and returns the partial trace.

In [24]:
def run_chain(steps):
    ctx, trace = {}, []
    for name, args in steps:
        a = args(ctx) if callable(args) else args
        res = csv_analyzer(**a)
        trace.append({"step": name, "ok": res["ok"]})
        if not res["ok"]:
            return {"ok": False, "failed_at": name, "trace": trace, "error": res["error"]}
        ctx[name] = res["data"]
    return {"ok": True, "trace": trace, "ctx": ctx}

chain = run_chain([
    ("profile",  {"path": SAMPLE}),
    ("group_by", {"path": SAMPLE, "action": "group_by", "by": "category", "metric": "amount", "agg": "sum"}),
])
if chain["ok"]:
    groups = chain["ctx"]["group_by"]["groups"]
    top, total = next(iter(groups.items())), sum(groups.values())
    print(f"Top category: {top[0]} = {top[1]} ({top[1] / total * 100:.1f}% of {total})")
    print("Data quality warnings:", *chain["ctx"]["profile"]["warnings"], sep="\n - ")
print(chain["trace"])

# failure path: second step fails, chain stops cleanly
bad = run_chain([("profile", {"path": SAMPLE}),
                 ("group_by", {"path": SAMPLE, "action": "group_by", "by": "city", "metric": "amount"})])
print("\nFailed chain ->", bad["failed_at"], "|", bad["error"]["message"])

Top category: Tech = 46800 (93.7% of 49930)
Data quality warnings:
 - 'amount' has 1 outlier(s) (IQR rule).
 - 1 duplicate row(s) found.
[{'step': 'profile', 'ok': True}, {'step': 'group_by', 'ok': True}]

Failed chain -> group_by | Column 'city' not found. Available: ['order_id', 'date', 'customer', 'category', 'amount', 'rating']


In [27]:
show("my file", csv_analyzer(path="C:\\Users\\Aalhad Ramteke\\Downloads\\heart.csv"))


=== my file ===
{
  "ok": true,
  "data": {
    "rows": 303,
    "columns_count": 14,
    "duplicate_rows": 1,
    "columns": {
      "age": {
        "dtype": "int64",
        "missing": 0,
        "missing_pct": 0.0,
        "unique": 41,
        "min": 29,
        "max": 77,
        "mean": 54.37,
        "median": 55.0,
        "std": 9.08,
        "outliers_iqr": 0
      },
      "sex": {
        "dtype": "int64",
        "missing": 0,
        "missing_pct": 0.0,
        "unique": 2,
        "min": 0,
        "max": 1,
        "mean": 0.68,
        "median": 1.0,
        "std": 0.47,
        "outliers_iqr": 0
      },
      "cp": {
        "dtype": "int64",
        "missing": 0,
        "missing_pct": 0.0,
        "unique": 4,
        "min": 0,
        "max": 3,
        "mean": 0.97,
        "median": 1.0,
        "std": 1.03,
        "outliers_iqr": 0
      },
      "trestbps": {
        "dtype": "int64",
        "missing": 0,
        "missing_pct": 0.0,
        "unique": 49,
  